# Running 21-cm simulations with primordial non-Gaussianity

This notebook walks through running `21cmFirstCLASS_NG` with a local primordial
non-Gaussianity of amplitude $f_{\rm NL}$, and comparing the result against the
Gaussian baseline.

It uses `runsims_FNL.py`, which sits alongside this notebook and wraps the
lightcone call, the caching, and the plotting.

**Before you start.** A single lightcone at the settings used here takes of
order an hour on a few cores, and the parameter grid in section 5 multiplies
that. Start with the smallest box, confirm the pipeline works end to end, and
only then scale up.

**Prerequisites**

* `21cmFirstCLASS_NG` installed (see the README)
* `classy`, the CLASS Python wrapper
* enough memory: with `NON_GAUSS_IC = True` the de-aliasing grid is
  `EXTRA_DIM_FNL` times `DIM` on a side, and memory scales as its cube

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import runsims_FNL as rs

# Where runs and figures are written. Nothing is created until you call this.
rs.set_output_folder('./fnl_runs/')

# Optional: LaTeX text rendering in the figures. Needs a LaTeX installation,
# and fails at draw time rather than here, so it is off by default.
# rs.use_latex_fonts()

plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

## 1. What the flags mean

Non-Gaussianity enters `21cmFirstCLASS_NG` in three independent places, and
each can be switched on separately. Running them one at a time is the only way
to see which one is responsible for a change in the signal.

| Flag | Where it acts | What it changes |
|---|---|---|
| `NON_GAUSS_IC` | initial conditions | the density field is built from a non-Gaussian potential, $\Phi = \Phi_G + f_{\rm NL}(\Phi_G^2 - \langle\Phi_G^2\rangle)$ |
| `NON_GAUSS_FCOLL_COND` | conditional halo mass function | the collapsed fraction in each cell |
| `NON_GAUSS_FCOLL_UNCOND` | unconditional halo mass function | the global ionizing emissivity |

Each correction also has a choice of approximation:

| Flag | Default | Alternative |
|---|---|---|
| `USE_EDG_uncond_hmf` | Edgeworth expansion ([arXiv:2009.01245](https://arxiv.org/abs/2009.01245)) | saddlepoint |
| `USE_LD_cond_hmf` | Lidz / D'Aloisio closed form | two-scale saddlepoint |
| `NG_MODEL_APPROX` | high-barrier limit ([arXiv:1304.8049](https://arxiv.org/abs/1304.8049)) | full expression ([arXiv:1206.3305](https://arxiv.org/abs/1206.3305)) |

The two approximations in each row agree at small $|f_{\rm NL}|$ and separate
where the expansion breaks down. **The difference between them is a useful
estimate of the modelling systematic, and worth quoting.**

### Two things that are easy to get wrong

**The random seed.** Comparing a Gaussian and a non-Gaussian run only isolates
the non-Gaussian effect if both use the same `random_seed`. With different
seeds the difference is dominated by cosmic variance. `runsims_FNL` fixes the
seed to 1 for exactly this reason.

**`KCUT_FNL` against the box.** Modes below $k_{\rm cut}$ stay Gaussian. If
$k_{\rm cut}$ is above the box fundamental mode $2\pi/L$, part of the
non-Gaussianity your box could represent is being removed. The next cell
checks this.

In [ ]:
Lbox = 300.     # comoving Mpc
Nbox = 100      # cells per side

k_fundamental = 2*np.pi / Lbox
k_nyquist = np.pi * Nbox / Lbox

print(f"box {Lbox:.0f} Mpc, {Nbox}^3 cells")
print(f"  fundamental mode  k_min = {k_fundamental:.4f} 1/Mpc")
print(f"  Nyquist           k_max = {k_nyquist:.4f} 1/Mpc")

for kcut in (1e-3, 1e-2, 1e-1, 1.0):
    if kcut < k_fundamental:
        verdict = "below the box: effectively scale-independent"
    elif kcut < k_nyquist:
        verdict = "inside the box: only modes above it are non-Gaussian"
    else:
        verdict = "ABOVE Nyquist: no box mode is non-Gaussian"
    print(f"  KCUT_FNL = {kcut:<6g} -> {verdict}")

## 2. A first pair of runs

We start with the Gaussian baseline and one non-Gaussian run. `import_stuff`
loads a cached result if it exists and runs the simulation otherwise, so
re-executing this cell is cheap once the runs are done.

`USE_MINI_HALOS = False` keeps Population III stars out, which cuts the runtime
substantially. Turn it on once the pipeline is working.

In [ ]:
fnl_test = 200.

common = dict(force_mmax=0.,          # no upper mass cut on the integrals
              kcut_fnl=1e-3,          # scale-independent for this box
              USE_MINI_HALOS=False,
              nongauss_IC=True,
              nongauss_fcollcond=True,
              nongauss_fcolluncond=True,
              use_lidz_approx=False,
              use_ld_cond=True,
              use_edg_uncond=True,
              max_epsilon=0.,
              extra_dim_fnl=1.5)      # Orszag 3/2 de-aliasing

Gauss = rs.import_stuff(Lbox, Nbox, fnl=0.,        **common)
NG    = rs.import_stuff(Lbox, Nbox, fnl=fnl_test,  **common)
NGm   = rs.import_stuff(Lbox, Nbox, fnl=-fnl_test, **common)

print("loaded three runs")

## 3. The global signal

The first thing to look at is the sky-averaged brightness temperature and the
neutral fraction. Positive $f_{\rm NL}$ raises the abundance of rare haloes,
which brings reionization forward; negative $f_{\rm NL}$ delays it.

If the two non-Gaussian curves sit on top of the Gaussian one, check first that
the simulation actually used the flags: a run saved with one set of parameters
and loaded with another silently returns the wrong file. The filename printed
by `run_filename` is the thing to verify.

In [ ]:
print(rs.run_filename(Lbox, Nbox, 0.,        **common))
print(rs.run_filename(Lbox, Nbox, fnl_test,  **common))
print(rs.run_filename(Lbox, Nbox, -fnl_test, **common))

In [ ]:
rs.plot_globals(Lbox, Nbox, [0., fnl_test, -fnl_test],
                force_mmax=common['force_mmax'],
                kcut_fnl=common['kcut_fnl'],
                USE_MINI_HALOS=common['USE_MINI_HALOS'])

## 4. The power spectrum

The 21-cm power spectrum is where a survey would actually look. Plotting the
ratio to the Gaussian run rather than the spectra themselves removes most of the
shape and leaves the non-Gaussian signal.

Because both runs share a random seed, the ratio is not dominated by cosmic
variance on large scales — but it is still a single realisation, so do not read
too much into a feature seen in one box.

In [ ]:
compare = [Gauss, NG, NGm]
labels = ['Gaussian',
          rf'$f_{{\rm NL}} = +{fnl_test:.0f}$',
          rf'$f_{{\rm NL}} = -{fnl_test:.0f}$']

rs.plot_compare_multi(k_vals=[0.1, 0.5],
                      compare=compare,
                      list_labels=labels)

In [ ]:
# Coeval slices: the Gaussian box and the difference from it
rs.plot_compare_boxes(0, Lbox, compare, list_labels=labels)

## 5. Isolating where the effect comes from

With all three flags on, a change in the signal could come from the initial
conditions, the conditional mass function, the unconditional one, or any
combination. Switching them on one at a time separates the contributions.

This is worth doing once per configuration you intend to publish. It is also
the cheapest way to catch a flag that is not doing anything: if turning one on
changes nothing at all, either it is genuinely negligible or it is not reaching
the C code.

In [ ]:
variants = {
    'IC only':          dict(nongauss_IC=True,  nongauss_fcollcond=False, nongauss_fcolluncond=False),
    'conditional only': dict(nongauss_IC=False, nongauss_fcollcond=True,  nongauss_fcolluncond=False),
    'unconditional only': dict(nongauss_IC=False, nongauss_fcollcond=False, nongauss_fcolluncond=True),
    'all three':        dict(nongauss_IC=True,  nongauss_fcollcond=True,  nongauss_fcolluncond=True),
}

runs = {}
for name, flags in variants.items():
    kw = dict(common); kw.update(flags)
    runs[name] = rs.import_stuff(Lbox, Nbox, fnl=fnl_test, **kw)
    print(f"  {name:20s} -> {rs.run_filename(Lbox, Nbox, fnl_test, **kw)}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

z_g, T_g = Gauss[0], Gauss[1]
for name, r in runs.items():
    ax.plot(r[0], np.array(r[1]) - np.interp(r[0], z_g, T_g), label=name)

ax.axhline(0., color='0.6', lw=.8)
ax.set(xlabel='redshift', ylabel=r'$\Delta \bar{T}_{21}$ [mK]',
       title=rf'difference from Gaussian, $f_{{\rm NL}} = {fnl_test:.0f}$')
ax.legend(); ax.grid(alpha=.3)
fig.tight_layout()

## 6. Choice of approximation as a systematic

The Edgeworth and saddlepoint schemes are two truncations of the same
expansion. Where both are valid they agree; where they separate, at least one
has broken down. Running both and quoting the difference is more honest than
picking one and hoping.

The same applies to `USE_LD_cond_hmf` on the conditional side.

In [ ]:
kw_edg = dict(common); kw_edg['use_edg_uncond'] = True
kw_sad = dict(common); kw_sad['use_edg_uncond'] = False

edg = rs.import_stuff(Lbox, Nbox, fnl=fnl_test, **kw_edg)
sad = rs.import_stuff(Lbox, Nbox, fnl=fnl_test, **kw_sad)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(edg[0], edg[1], label='Edgeworth (arXiv:2009.01245)')
ax.plot(sad[0], sad[1], '--', label='saddlepoint')
ax.plot(z_g, T_g, ':', color='0.5', label='Gaussian')
ax.set(xlabel='redshift', ylabel=r'$\bar{T}_{21}$ [mK]',
       title=rf'unconditional HMF scheme, $f_{{\rm NL}} = {fnl_test:.0f}$')
ax.legend(); ax.grid(alpha=.3)
fig.tight_layout()

spread = np.max(np.abs(np.array(edg[1]) - np.array(sad[1])))
print(f"largest difference between the two schemes: {spread:.3f} mK")

## 7. A grid of runs

`run_many` loops over the parameter grid and saves each run. Everything is
cached, so a later `import_stuff` with the same parameters returns immediately.

Mind the combinatorics: the number of lightcones is the product of all five
list lengths. The call below is 4 runs; adding one more value to two of the
lists would make it 16.

In [ ]:
# Uncomment to run. Expect roughly an hour per lightcone at these settings.
#
# rs.run_many(Lbox=Lbox, Nbox=Nbox,
#             fnl_arr=[0., 100., 200., -200.],
#             force_mmax_arr=[0.],
#             kcut_fnl_arr=[1e-3],
#             use_MUSE_MINI_HALOS_list=[False],
#             nongauss_IC=True,
#             nongauss_fcollcond=True,
#             nongauss_fcolluncond=True,
#             extra_dim_fnl=1.5,
#             nthreads=6)

## 8. Diagnostics when something looks wrong

`WRITE_CGF_DIAG` makes the C code write one binary record per evaluation of the
non-Gaussian correction, one file per OpenMP thread, into `files/` relative to
the working directory. **The directory must exist first**, or the records are
discarded with a warning.

This is slow and produces a lot of data, so use it on a small box only. What it
is good for is finding out *where* in the $(M, z)$ plane the expansion stops
behaving: the per-order magnitudes and the saddlepoint discriminant are both in
the record.

In [ ]:
import os
os.makedirs('files', exist_ok=True)

# Record layout mirrors CGFDiag in src/ps.c. If you add a field there, add it here.
cgf_dtype = np.dtype([
    ('epsilon',    'f8'),   # |kappa3 H3(nu)/6|, size of the leading correction
    ('Dval',       'f8'),   # saddlepoint discriminant S^2 + 2 mu3 barrier
    ('exponent',   'f8'),   # saddlepoint exponent
    ('validation', 'f8'),   # mu3 t / (3 S), third order against second
    ('M',          'f8'),
    ('z',          'f8'),
    ('S',          'f8'),   # variance sigma^2(M)
    ('mu3',        'f8'),
    ('t',          'f8'),   # saddlepoint
    ('ratio_CGF',  'f8'),   # correction actually applied
])


def read_cgf_diag(pattern='files/dNion_diag_thread_*.bin'):
    '''Concatenate the per-thread diagnostic files into one array.'''
    import glob
    files = sorted(glob.glob(pattern))
    if not files:
        print(f'no diagnostic files matching {pattern}')
        return np.empty(0, dtype=cgf_dtype)
    data = np.concatenate([np.fromfile(f, dtype=cgf_dtype) for f in files])
    print(f'read {len(data)} records from {len(files)} thread files')
    return data


# d = read_cgf_diag()
# if len(d):
#     plt.scatter(d['M'], d['z'], c=np.log10(np.abs(d['ratio_CGF'])), s=2)
#     plt.xscale('log'); plt.colorbar(label=r'$\log_{10}|{\rm ratio}|$')
#     plt.xlabel(r'$M\ [M_\odot]$'); plt.ylabel('z')

## Checklist before publishing a run

| Check | Why |
|---|---|
| same `random_seed` across compared runs | otherwise the difference is cosmic variance |
| `KCUT_FNL` below $2\pi/L_{\rm box}$, or deliberately above it | decides which modes carry the non-Gaussianity |
| `F_NL = 0` reproduces the Gaussian code | confirms nothing leaks in when it should not |
| both approximation schemes run | their difference is the modelling systematic |
| flags switched on one at a time | shows which term drives the signal |
| `EXTRA_DIM_FNL = 1.5` for production | below this the squared field aliases |
| convergence of the Edgeworth series checked | see `notebook_fNL_collapse_fraction.ipynb` |

One further point worth stating in any paper using this code: because the
Gaussian baseline here is `21cmFirstCLASS` compiled in double precision rather
than single, it does not reproduce the published single-precision results
bit-for-bit even at `F_NL = 0`. See the README.